# batchnorm-running-stats — worked example 1: Apply two sequential EMA updates to running stats

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `batchnorm-running-stats`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

BatchNorm keeps a `running_mean` and `running_var` that are NOT learned by gradient descent — they are exponential moving averages of the per-channel batch statistics seen during training. Each training step blends the current batch's stats into the buffers via `running = (1 - momentum) * running + momentum * batch`. Because the update is a recurrence, running two batches in sequence applies the blend twice, so the second result depends on the first.

## Worked solution

**Goal.** Start from default buffers (`running_mean = 0`, `running_var = 1`) and feed two batches in order, returning the buffers after each.

**Step 1 — per-channel batch stats.** For each batch of shape `(B, C, H, W)`, reduce over the batch and spatial axes `(0, 2, 3)` so one number survives per channel. We use `unbiased=True` for the variance because the *running* variance is meant to estimate the population variance — this matches PyTorch's choice for the buffer (even though the in-batch normalize step uses the biased `/N` variance).

**Step 2 — first EMA blend.** Apply `running = (1 - m) * running + m * batch`. With `m = 0.1`, the buffer keeps 90% of its old value and absorbs 10% of the new batch stat. This is why a single batch barely moves the buffer early on.

**Step 3 — second EMA blend.** The key insight: we feed the *updated* buffer back in. The recurrence compounds, so after batch 2 the mean is `(1-m)^2 * 0 + (1-m)*m*mean1 + m*mean2`. We don't compute that closed form — we just call the update twice and let the recurrence handle it.

**Why it works.** EMA is a leaky integrator: old information decays geometrically by `(1-m)` per step. Running the update function twice is exactly two steps of that integrator, which is what training does across many minibatches.

In [ ]:
def two_step_ema(batches, momentum):
    running_mean = t.zeros(batches[0].shape[1])
    running_var = t.ones(batches[0].shape[1])
    snapshots = []
    for x in batches:
        batch_mean = x.mean(dim=(0, 2, 3))
        batch_var = x.var(dim=(0, 2, 3), unbiased=True)
        running_mean = (1 - momentum) * running_mean + momentum * batch_mean
        running_var = (1 - momentum) * running_var + momentum * batch_var
        snapshots.append((running_mean.clone(), running_var.clone()))
    return snapshots

t.manual_seed(0)
b1 = t.randn(4, 3, 5, 5) + 2.0
b2 = t.randn(4, 3, 5, 5) - 1.0
snaps = two_step_ema([b1, b2], momentum=0.1)
print("after batch 1 mean:", snaps[0][0].round(decimals=4))
print("after batch 2 mean:", snaps[1][0].round(decimals=4))
print("after batch 2 var :", snaps[1][1].round(decimals=4))